In [4]:
!pip install koreanize_matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 51.5 MB/s eta 0:00:00


In [6]:
import warnings
import koreanize_matplotlib
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go


# 경고 무시
warnings.filterwarnings("ignore")
%config lnlineBackend.figure_format = 'retina'

pd.set_option('display.max_columns', None)

pd.set_option('display.width', 1000)  # 출력할 너비를 넉넉하게 조정
pd.set_option('display.expand_frame_repr', False)  # 옆으로 길어져도 줄바꿈 없이 출력
pd.set_option('display.max_colwidth', None)  # 긴 문자열도 생략 없이 출력

In [7]:
# 구글 마운트하기
from google.colab import drive
drive.mount('/content/drive')


# 현재 경로 지정하기
import os
os.chdir('/content/drive/MyDrive/파트4')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
API_KEY_PATH = '/content/drive/MyDrive/파트4/sprintda03-yujin.json'

def get_df(db_name, table_name):
    table_name = pd.read_csv(
        f"gs://high_project/{db_name}/{table_name}.csv",
        storage_options={'token' : API_KEY_PATH}
        )
    return table_name

In [10]:
accounts_blockrecord = get_df('votes','accounts_blockrecord')
accounts_blockrecord = accounts_blockrecord[['id', 'user_id', 'block_user_id', 'reason', 'created_at']]
accounts_blockrecord = accounts_blockrecord.loc[:, 'user_id':].drop_duplicates()
accounts_blockrecord['created_at'] = pd.to_datetime(accounts_blockrecord['created_at'])

## info

In [11]:
accounts_blockrecord.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19477 entries, 0 to 19481
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   user_id        19477 non-null  int64         
 1   block_user_id  19477 non-null  int64         
 2   reason         19477 non-null  object        
 3   created_at     19477 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(1)
memory usage: 760.8+ KB


## 중복값

In [ ]:
accounts_blockrecord.loc[:, 'user_id':].duplicated().sum()

## 결측값

In [ ]:
accounts_blockrecord.isna().sum()

## reason에 따른 신고받은수 분석

In [12]:
accounts_blockrecord['reason'].unique()

array(['그냥...', '친구 사이가 어색해짐', '나랑 관련 없는 질문을 자꾸 보냄', '기타', '모르는 사람임',
       '너무 많은 양의 질문을 보냄', '사칭 계정'], dtype=object)

### total

In [48]:
total_block = accounts_blockrecord.groupby(['block_user_id'])['user_id'].nunique().reset_index()
fig = px.box(total_block, x='user_id', title='유저당 신고누적수 분포')
fig.update_layout(
    width=1200,   # 너비
    height=300   # 높이
)
fig.show()

In [81]:
total_block.sort_values(by='user_id', ascending=False)
total_block.query('user_id > 20')

,block_user_id,user_id
565,877266,24
1037,897681,21
12462,1380465,25
12804,1395312,24


In [80]:
accounts_blockrecord.query('block_user_id == 1380465').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
모르는 사람임,21
사칭 계정,2
친구 사이가 어색해짐,2


In [79]:
accounts_blockrecord.query('block_user_id == 877266').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
모르는 사람임,24


In [78]:
accounts_blockrecord.query('block_user_id == 1395312').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
모르는 사람임,23
친구 사이가 어색해짐,1


In [77]:
accounts_blockrecord.query('block_user_id == 897681').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
나랑 관련 없는 질문을 자꾸 보냄,2
너무 많은 양의 질문을 보냄,4
모르는 사람임,16


### stranger

In [53]:
strangers = accounts_blockrecord.query('reason == "모르는 사람임"').groupby(['block_user_id'])['user_id'].nunique().reset_index()
fig = px.box(strangers, x='user_id', title='유저당 모르는사람으로 신고누적수 분포')
fig.update_layout(
    width=1200,   # 너비
    height=300   # 높이
)
fig.show()

In [65]:
strangers.query('user_id > 20')

,block_user_id,user_id
297,877266,24
6251,1380465,21
6448,1395312,23


In [76]:
# 1. 877266
accounts_blockrecord.query('block_user_id == 877266').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
모르는 사람임,24


In [75]:
# 2. 1395312
accounts_blockrecord.query('block_user_id == 1395312').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
모르는 사람임,23
친구 사이가 어색해짐,1


In [74]:
# 3. 1380465
accounts_blockrecord.query('block_user_id == 1380465').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
모르는 사람임,21
사칭 계정,2
친구 사이가 어색해짐,2


### imperson

In [55]:
imperson = accounts_blockrecord.query('reason == "사칭 계정"').groupby(['block_user_id'])['user_id'].nunique().reset_index()
fig = px.box(imperson, x='user_id', title='유저당 사칭계정으로 신고누적수 분포')
fig.update_layout(
    width=1200,   # 너비
    height=300   # 높이
)
fig.show()

In [69]:
imperson.query('user_id == 14')

,block_user_id,user_id
855,1198628,14


In [73]:
accounts_blockrecord.query('block_user_id == 1198628').groupby(['reason'])['user_id'].nunique()

,user_id
reason,
너무 많은 양의 질문을 보냄,1
모르는 사람임,4
사칭 계정,14
